In [2]:
import torch
import torch.nn as nn

import numpy as np

# 単一の配列
embedding_matrix = np.load('embedding_matrix.npy')

# .npz から取り出す場合
# data = np.load('embeddings.npz')
# embedding_matrix = data['embedding_matrix']


class AverageEmbeddingClassifier(nn.Module):
    def __init__(self, embedding_matrix):
        super().__init__()
        vocab_size, embedding_dim = embedding_matrix.shape

        self.embedding = nn.Embedding.from_pretrained(
            torch.tensor(embedding_matrix, dtype=torch.float),
            freeze=False,  # 埋め込みを学習可能にするならFalse
            padding_idx=0
        )
        self.linear = nn.Linear(embedding_dim, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, input_ids):
        """
        input_ids: (batch_size, seq_len)
        """
        embedded = self.embedding(input_ids)  # (batch, seq_len, emb_dim)
        mask = (input_ids != 0).unsqueeze(-1)  # padding無視用のマスク
        summed = (embedded * mask).sum(dim=1)  # 総和（paddingを除く）
        lengths = mask.sum(dim=1).clamp(min=1)  # 長さ（0除算回避）
        mean_emb = summed / lengths            # 平均埋め込みベクトル

        logits = self.linear(mean_emb)         # 線形変換
        probs = self.sigmoid(logits)           # 出力は確率 (0~1)
        return probs

# すでに作成済みの埋め込み行列を使う
# 例：embedding_matrix.shape = (50001, 300)

model = AverageEmbeddingClassifier(embedding_matrix)


# input_ids: (batch_size, seq_len)
# 例：pad済みバッチ
batch_input_ids = torch.nn.utils.rnn.pad_sequence(
    [d['input_ids'] for d in train_data[:8]], batch_first=True, padding_value=0
)

# 順伝播
with torch.no_grad():
    probs = model(batch_input_ids)
print(probs.squeeze())  # (batch_size,)

criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

# 例：1ステップの訓練
labels = torch.cat([d['label'] for d in train_data[:8]])  # (batch_size,)
optimizer.zero_grad()
outputs = model(batch_input_ids).squeeze()               # (batch_size,)
loss = criterion(outputs, labels)
loss.backward()
optimizer.step()


NameError: name 'train_data' is not defined

In [3]:
import torch
import torch.nn as nn
import numpy as np

# 事前に保存しておいた embedding_matrix を読み込む
embedding_matrix = np.load('embedding_matrix.npy')  # shape=(vocab_size, emb_dim)

class AvgEmbedLogisticRegression(nn.Module):
    def __init__(self, embedding_matrix, freeze_embeddings=False):
        super().__init__()
        vocab_size, emb_dim = embedding_matrix.shape

        # 埋め込みレイヤー
        self.embedding = nn.Embedding.from_pretrained(
            torch.tensor(embedding_matrix, dtype=torch.float),
            freeze=freeze_embeddings,
            padding_idx=0
        )

        # ロジスティック回帰のパラメータを自前で定義
        # weight: (emb_dim,), bias: scalar
        self.weight = nn.Parameter(torch.zeros(emb_dim))
        self.bias   = nn.Parameter(torch.zeros(1))

    def forward(self, input_ids):
        """
        input_ids: LongTensor of shape (batch_size, seq_len)
        """
        # 1) 埋め込み取得 → (batch, seq_len, emb_dim)
        emb = self.embedding(input_ids)

        # 2) padding を除くマスク → (batch, seq_len, 1)
        mask = (input_ids != 0).unsqueeze(-1).float()

        # 3) 要素ごとの積 → padding 部分はゼロ化
        emb_masked = emb * mask

        # 4) 各文の平均ベクトル → (batch, emb_dim)
        summed    = emb_masked.sum(dim=1)                   # (batch, emb_dim)
        lengths   = mask.sum(dim=1).clamp(min=1)            # (batch, 1)
        avg_emb   = summed / lengths

        # 5) ロジスティック回帰：σ(w⋅avg_emb + b)
        logits    = avg_emb.matmul(self.weight) + self.bias # (batch,)
        probs     = torch.sigmoid(logits)                   # (batch,)
        return probs

# モデル準備
model     = AvgEmbedLogisticRegression(embedding_matrix, freeze_embeddings=False)
criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

# --- 1ステップ訓練例 ---
# train_data は dict のリストで、各要素が {'input_ids': LongTensor, 'label': FloatTensor}
batch = train_data[:8]
batch_input_ids = torch.nn.utils.rnn.pad_sequence(
    [d['input_ids'] for d in batch], batch_first=True, padding_value=0
)  # (8, seq_len)
batch_labels = torch.cat([d['label'] for d in batch])  # (8,)

# 順伝播＋損失計算＋逆伝播＋更新
optimizer.zero_grad()
outputs = model(batch_input_ids)         # (8,)
loss    = criterion(outputs, batch_labels)
loss.backward()
optimizer.step()

print(f"Batch loss: {loss.item():.4f}")


NameError: name 'train_data' is not defined

In [4]:
# run_bow_classifier.py

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.nn.utils.rnn import pad_sequence

# ───────────────────────────────────────────────
# 1. 埋め込み行列の読み込み (70)
# ───────────────────────────────────────────────
# shape = (vocab_size, emb_dim)
embedding_matrix = np.load('embedding_matrix.npy')

# ───────────────────────────────────────────────
# 2. SST データ読み込み関数 (71)
# ───────────────────────────────────────────────
def load_sst(path_tsv, token2id):
    df = pd.read_csv(path_tsv, sep='\t', header=None, names=['label','text'])
    examples = []
    for _, row in df.iterrows():
        words = row['text'].split()
        ids = [token2id[w] for w in words if w in token2id]
        if not ids:
            continue
        examples.append({
            'input_ids': torch.tensor(ids, dtype=torch.long),
            'label':     torch.tensor([float(row['label'])], dtype=torch.float)
        })
    return examples

# 70 で構築済みの token2id をロードまたは定義しておく
# ここでは embedding_matrix.npy と同じ語彙順序で保存した
# token2id.npy を使う例
token2id = np.load('token2id.npy', allow_pickle=True).item()

train_data = load_sst('train.tsv', token2id)
dev_data   = load_sst('dev.tsv',   token2id)

# ミニバッチ作成ヘルパー
def make_batch(batch):
    input_ids = pad_sequence([ex['input_ids'] for ex in batch],
                             batch_first=True, padding_value=0)
    labels    = torch.cat([ex['label'] for ex in batch]).squeeze()
    return input_ids, labels

# ───────────────────────────────────────────────
# 3. モデル定義：平均埋め込み＋ロジスティック回帰 (72)
# ───────────────────────────────────────────────
class BoWClassifier(nn.Module):
    def __init__(self, embedding_matrix, freeze_embeddings=False):
        super().__init__()
        vocab_size, emb_dim = embedding_matrix.shape
        self.embedding = nn.Embedding.from_pretrained(
            torch.tensor(embedding_matrix, dtype=torch.float),
            freeze=freeze_embeddings,
            padding_idx=0
        )
        self.weight = nn.Parameter(torch.zeros(emb_dim))
        self.bias   = nn.Parameter(torch.zeros(1))

    def forward(self, input_ids):
        emb     = self.embedding(input_ids)                   # (B, L, D)
        mask    = (input_ids != 0).unsqueeze(-1).float()      # (B, L, 1)
        summed  = (emb * mask).sum(dim=1)                     # (B, D)
        lengths = mask.sum(dim=1).clamp(min=1)                # (B, 1)
        avg_emb = summed / lengths                           # (B, D)
        logits  = avg_emb.matmul(self.weight) + self.bias     # (B,)
        return torch.sigmoid(logits)                         # (B,)

# モデル、最適化手法、損失関数の準備
model     = BoWClassifier(embedding_matrix, freeze_embeddings=False)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.BCELoss()

# ───────────────────────────────────────────────
# 4. 1 エポック分の訓練ループ（例）とモデル保存
# ───────────────────────────────────────────────
model.train()
for i in range(0, len(train_data), 32):
    batch = train_data[i:i+32]
    x, y  = make_batch(batch)
    optimizer.zero_grad()
    pred = model(x)
    loss = criterion(pred, y)
    loss.backward()
    optimizer.step()
    if i % 320 == 0:
        print(f"[train] step {i:4d}/{len(train_data)}  loss={loss.item():.4f}")

# モデル保存
torch.save(model.state_dict(), 'bow_classifier.pth')
print("Model saved to bow_classifier.pth")

# ───────────────────────────────────────────────
# 5. 開発セットでの簡易評価
# ───────────────────────────────────────────────
model.eval()
correct = 0
with torch.no_grad():
    for i in range(0, len(dev_data), 32):
        batch = dev_data[i:i+32]
        x, y  = make_batch(batch)
        pred = model(x).round()
        correct += (pred == y).sum().item()
acc = correct / len(dev_data)
print(f"[dev] accuracy = {acc:.4f}")


FileNotFoundError: [Errno 2] No such file or directory: 'token2id.npy'

In [6]:
# run_bow_full.py

import numpy as np
import pandas as pd
import gensim
import torch
import torch.nn as nn
from torch.nn.utils.rnn import pad_sequence

# ───────────────────────────────────────────────
# 1. 事前学習済み単語埋め込みの読み込み・辞書作成 (70)
# ───────────────────────────────────────────────
print("Loading pretrained embeddings...")
w2v_path = 'C:/Users/eriya/Documents/classcontents/25春夏/100knock2025/chapter06/GoogleNews-vectors-negative300.bin.gz'
w2v = gensim.models.KeyedVectors.load_word2vec_format(w2v_path, binary=True)

# 利用語彙を制限（例: 上位100,000語）
vocab_list = list(w2v.key_to_index.keys())[:10000]
vocab_size = len(vocab_list) + 1    # +1 for <PAD>
emb_dim     = w2v.vector_size

# 埋め込み行列と双方向辞書
print("Building embedding matrix and token2id...")
embedding_matrix = np.zeros((vocab_size, emb_dim), dtype=np.float32)
token2id = {'<PAD>': 0}
id2token = {0: '<PAD>'}

for idx, token in enumerate(vocab_list, start=1):
    token2id[token]       = idx
    id2token[idx]         = token
    embedding_matrix[idx] = w2v[token]

# ───────────────────────────────────────────────
# 2. SST データセット読み込み・前処理 (71)
# ───────────────────────────────────────────────
def load_sst(path, token2id):
    df = pd.read_csv(path, sep='\t', header=None, names=['label','text'])
    examples = []
    for _, row in df.iterrows():
        words = row['text'].split()
        ids   = [token2id[w] for w in words if w in token2id]
        if not ids:  # 全単語がOOVならスキップ
            continue
        examples.append({
            'input_ids': torch.tensor(ids, dtype=torch.long),
            'label':     torch.tensor([float(row['label'])], dtype=torch.float)
        })
    return examples

print("Loading SST train/dev...")
train_data = load_sst('train.tsv', token2id)
dev_data   = load_sst('dev.tsv',   token2id)

def make_batch(batch):
    x = pad_sequence([ex['input_ids'] for ex in batch],
                     batch_first=True, padding_value=0)
    y = torch.cat([ex['label'] for ex in batch]).squeeze()
    return x, y

# ───────────────────────────────────────────────
# 3. モデル定義：平均埋め込み＋ロジスティック回帰 (72)
# ───────────────────────────────────────────────
class BoWClassifier(nn.Module):
    def __init__(self, embedding_matrix, freeze_embeddings=False):
        super().__init__()
        vocab_size, emb_dim = embedding_matrix.shape
        self.embedding = nn.Embedding.from_pretrained(
            torch.tensor(embedding_matrix), freeze=freeze_embeddings, padding_idx=0
        )
        self.weight = nn.Parameter(torch.zeros(emb_dim))
        self.bias   = nn.Parameter(torch.zeros(1))

    def forward(self, input_ids):
        emb     = self.embedding(input_ids)                    # (B,L,D)
        mask    = (input_ids != 0).unsqueeze(-1).float()       # (B,L,1)
        summed  = (emb * mask).sum(dim=1)                      # (B,D)
        lengths = mask.sum(dim=1).clamp(min=1)                 # (B,1)
        avg_emb = summed / lengths                            # (B,D)
        logits  = avg_emb.matmul(self.weight) + self.bias      # (B,)
        return torch.sigmoid(logits)                          # (B,)

# ───────────────────────────────────────────────
# 4. 訓練ループ・モデル保存・評価
# ───────────────────────────────────────────────
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model     = BoWClassifier(embedding_matrix, freeze_embeddings=False).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.BCELoss()

# 1 エポック分の訓練例
print("Training...")
model.train()
for i in range(0, len(train_data), 32):
    batch = train_data[i:i+32]
    x, y  = make_batch(batch)
    x, y  = x.to(device), y.to(device)
    optimizer.zero_grad()
    pred = model(x)
    loss = criterion(pred, y)
    loss.backward()
    optimizer.step()
    if i % 320 == 0:
        print(f"  step {i}/{len(train_data)}  loss={loss.item():.4f}")

# 保存
torch.save(model.state_dict(), 'bow_classifier.pth')
print("Model saved -> bow_classifier.pth")

# 簡易評価
print("Evaluating on dev set...")
model.eval()
correct = 0
with torch.no_grad():
    for i in range(0, len(dev_data), 32):
        batch = dev_data[i:i+32]
        x, y  = make_batch(batch)
        x, y  = x.to(device), y.to(device)
        pred = model(x).round()
        correct += (pred == y).sum().item()
acc = correct / len(dev_data)
print(f"Dev accuracy: {acc:.4f}")


Loading pretrained embeddings...
Building embedding matrix and token2id...
Loading SST train/dev...


ValueError: could not convert string to float: 'sentence'

In [7]:
import numpy as np
import torch
import torch.nn as nn
import pickle
from torch.nn.utils.rnn import pad_sequence

# データと埋め込みロード
embedding_matrix = np.load('70_embeddings.npy')
with open('71_sst_data.pkl', 'rb') as f: data = pickle.load(f)
train_data = data['train']

def make_batch(batch):
    x = pad_sequence([ex['input_ids'] for ex in batch], batch_first=True, padding_value=0)
    y = torch.cat([ex['label'] for ex in batch]).squeeze()
    return x, y

# モデル定義
class BoWClassifier(nn.Module):
    def __init__(self, embedding_matrix):
        super().__init__()
        self.embedding = nn.Embedding.from_pretrained(
            torch.tensor(embedding_matrix), freeze=False, padding_idx=0)
        emb_dim = embedding_matrix.shape[1]
        self.weight = nn.Parameter(torch.zeros(emb_dim))
        self.bias   = nn.Parameter(torch.zeros(1))
    def forward(self, input_ids):
        emb    = self.embedding(input_ids)
        mask   = (input_ids!=0).unsqueeze(-1).float()
        summed = (emb*mask).sum(dim=1)
        lengths= mask.sum(dim=1).clamp(min=1)
        avg    = summed/lengths
        logits = avg.matmul(self.weight) + self.bias
        return torch.sigmoid(logits)

model = BoWClassifier(embedding_matrix)
# 保存
torch.save(model.state_dict(), '72_bow_init.pth')